## Configuration

In [ ]:
from pathlib import Path
import tomllib
from utils.utils import *
import json

PATH_EEG = Path("../EEG/EEG/results/")
FILENAME = 'Case_07_ExG.csv'
MARKER_FILE = 'Case_07_Marker.csv'
CONFIG_PATH = "../EEG/EEG/config/config.toml"

with open(CONFIG_PATH, "rb") as f:
    config = tomllib.load(f)

BANDS = config['BANDS']
SEGMENT_DEF = config['SEGMENT_DEF']
RANDOM_SEED = config['RANDOM_SEED']
DURATION = config['DURATION']

SFREQ = config['SFREQ']
L_FREQ = config['L_FREQ']
H_FREQ = config['H_FREQ']

## Raw Data

In [ ]:
raw_full, df_full, df_markers_full = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

raws, labels = extract_segments(raw_full, df_full, df_markers_full, SEGMENT_DEF)

In [ ]:
plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Theta')

## Preprocessing

In [ ]:
import mne
from mne.preprocessing import ICA
raw, df, df_markers = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

# ICA
# n_components=15 (16 Channels, ICA finding max n_channels - 1 or same number)
ica = ICA(n_components=15, max_iter='auto', random_state=RANDOM_SEED)

print(">>> ICA (Might take a while)...")
ica.fit(raw)

print(">>> Showing Components...")
ica.plot_components()
plt.show()

ica.plot_sources(raw, show_scrollbars=True)
plt.show()

In [ ]:
exclude_indices = [0, 1, 2, 4, 14] 

print(f">>> Removing ingredients: {exclude_indices}")
ica.exclude = exclude_indices

raw_clean = raw.copy()
ica.apply(raw_clean)

print(">>> Cleaned data.")

target_chan = 'F3' if 'F3' in raw.ch_names else raw.ch_names[0]

# print(f"Comparison on the channel {target_chan}...")
# fig, ax = plt.subplots(figsize=(15, 6))
# ax.plot(raw.get_data(picks=target_chan)[0, 1000:2500], color='red', alpha=0.5, label='Before cleaning (Original)')
# ax.plot(raw_clean.get_data(picks=target_chan)[0, 1000:2500], color='black', label='After cleaning (Clean)')
# ax.set_title(f"Artifact Removal Effect (Eyes and Muscles) - Channel {target_chan}")
# ax.legend()
# plt.show()

In [ ]:
print(f"Comparison on the channel {target_chan}...")

data_orig_uv = raw.get_data(picks=target_chan)[0, 1000:2500] * 1e6
data_clean_uv = raw_clean.get_data(picks=target_chan)[0, 1000:2500] * 1e6

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

ax1.plot(data_orig_uv, color='red', alpha=0.7, label='Before cleaning (Original)')
ax1.set_title(f"Artifact Removal Effect - Channel {target_chan}\nOriginal Signal with Macro-Artifact", fontsize=14)
ax1.set_ylabel("Amplitude (µV)")
ax1.legend(loc="upper right")
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot(data_clean_uv, color='black', label='After cleaning (Clean)')
ax2.set_title("Cleaned Signal - Preserved Brainwaves", fontsize=14)
ax2.set_xlabel("Time (Samples)")
ax2.set_ylabel("Amplitude (µV)")
ax2.legend(loc="upper right")
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
raws, labels = extract_segments(raw_clean, df_full, df_markers_full, SEGMENT_DEF)

plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Gamma')

## 1. Analiza Pasma Alpha (8–12 Hz): Asymetria i Sterowanie
* **Dominacja Prawej Kory Potylicznej (O2/O1):** Na wszystkich mapach Alpha wyraźnie widać potężną, czerwoną aktywność z tyłu głowy, z bardzo silnym przesunięciem na prawą półkulę (obszar O2).
* **Faza 1 vs Faza 2:**
    * **Faza 1 (Brainrot):** Silna, skondensowana plama Alpha na potylicy. Mózg wchodzi w klasyczny tryb "Zombie" – patrzy w ekran, ale kora wzrokowa nie przetwarza aktywnie treści. Świadomość wizualna jest wyłączona.
    * **Faza 2 (Smart):** Następuje drastyczny spadek mocy Alpha (kolor zmienia się na jasnopomarańczowy/biały). Treść edukacyjna wybudza mózg ze stanu transu, zmuszając korę wzrokową do ponownego zaangażowania się.
    * **Wpływ Scrollowania (Fazy 3 i 4):** Włączenie mechaniki scrollowania (Brainrot Scroll i Smart Scroll) powoduje powrót silnej aktywności Alpha (czerwień). Sugeruje to, że sam akt przewijania kciukiem staje się rutynową, zautomatyzowaną czynnością, która wręcz ułatwia pacjentowi powrót do biernego "odpływania", niezależnie od tego, czy ogląda treści Smart czy Brainrot.

## 2. Analiza Pasma Beta (13–30 Hz): "Skanowanie Przestrzenne"
* **Obserwacja Asymetrii:** Aktywność Beta (odpowiadająca za aktywne skupienie i rozwiązywanie problemów) jest całkowicie wyciszona w płatach czołowych, a zogniskowana wyłącznie na prawej potylicy (O2).
* **Wniosek:** Prawa kora potyliczna odpowiada za przetwarzanie przestrzenne, kolory i szybki ruch. Silna Beta w tym obszarze sugeruje, że pacjent intensywnie "skanuje" migające przebodźcowane ekrany, próbując wyłapać układ przestrzenny wideo, zamiast głęboko analizować jego logiczną treść językową (co powinno aktywować lewą półkulę).

## 3. Analiza Pasma Delta (1–4 Hz): Głębokość Transu i Izolacja
* **Globalne Wyciszenie:** Fale Delta (związane z głębokim snem lub silnym odcięciem sensorycznym) dominują w tylnej części czaszki podczas niemal wszystkich faz.
* **Hierarchia Odcięcia:**
    1.  **Faza 1** Potężna, skondensowana czerwień na O2. Pacjent jest całkowicie "zahipnotyzowany". Mózg odcina zewnętrzne bodźce, generując fale typowe dla głębokiej izolacji poznawczej.
    2.  **Faza 3** Plama Delta wyraźnie blednie i kurczy się. Trudniejsza treść rozbija barierę transu, zmuszając pacjenta do "powrotu do rzeczywistości"..
    3.  **Faza 4** Delta ponownie silnie wybija w górę.

## 4. Analiza Pasma Gamma (>30 Hz): Koszt Poznawczy i "Przebodźcowanie"
* **Lokalizacja:** Gamma wykazuje identyczny wzorzec jak Delta i Alpha – ostra, ogniskowa aktywacja prawej kory potylicznej. W kontekście zadań wzrokowych, Gamma oznacza ekstremalnie szybkie, synchroniczne "zszywanie" obrazów w jedną całość (tzw. feature binding).
* **Interpretacja:** Najsilniejsza Gamma pojawia się w fazie Brainrot bez scrollowania. Oznacza to, że chaotyczny, szybko zmieniający się montaż typowy dla tego typu treści narzuca potężny koszt fizjologiczny na układ wzrokowy pacjenta.
* **Dominacja Fazy 4:** Co ciekawe, w fazie Smart (bez scrollowania) ten wysiłek drastycznie spada – statyczne, edukacyjne wideo daje korze wzrokowej czas na oddech..

---

1.  **Dominacja Przetwarzania Obrazowego":** Odbiera internet czysto wizualnie i przestrzennie. Brak jakiejkolwiek aktywności na lewej półkuli (O1, P3, F3 – odpowiedzialnych za logikę i tekst) we wszystkich pasmach dowodzi, że pacjent nie czyta napisów ani nie analizuje głęboko treści mówionych. Skupia się na kolorach, ruchu i bodźcach z prawej półkuli.
2.  **Trans Brainrotu:** Najsilniejsze dowody na "Zombie Mode" widać w Fazie 1 (Brainrot). Jednoczesny gigantyczny wystrzał fal Alpha (wyłączenie uwagi) i Delta (głębokie odcięcie) na potylicy udowadnia, że bierne oglądanie tego typu treści wprowadza mózg w stan fizjologicznego transu.
3.  **Scrollowanie:** Treści "Smart" wybudzają pacjenta (spadek Alpha i Delta). Jednak gdy dołożymy do nich konieczność scrollowania (Faza 4), pacjent z powrotem "odpływa". Sam fizyczny, powtarzalny ruch kciukiem pomaga badanemu obciążyć się poznawczo i wrócić do transu.

## Spektogram

In [ ]:
# Alpha (fatigue/relax)
plot_band_power_over_time(raws, labels, band=(8, 12), band_name='Alpha')
# Beta (Focus)
plot_band_power_over_time(raws, labels, band=(13, 30), band_name='Beta')

In [ ]:
# Emotion
calculate_asymmetry(raws, labels, ch_left='F3', ch_right='F4')

# According to Heller's model, parietal asymmetry is responsible for physiological arousal
calculate_asymmetry(raws, labels, ch_left='P3', ch_right='P4')

# Image processing method.
calculate_asymmetry(raws, labels, ch_left='O1', ch_right='O2')

# Motor
calculate_asymmetry(raws, labels, ch_left='C3', ch_right='C4')

### Podsumowanie

**F4/F3 (Motywacja i Zaangażowanie)**: Konsekwentnie ujemne wartości (od -0.48 do -0.65) oznaczają poznawcze wycofanie (Withdrawal). Pacjent nie wykazuje entuzjazmu ani motywacji w żadnej fazie – biernie przyjmuje to, co widzi. Co ciekawe, dodanie mechaniki scrollowania (zwłaszcza Smart Scroll) wyraźnie wyciąga go w stronę zera. Sam ruch palcem daje mu poczucie sprawczości i lekko go wybudza.

**P4/P3 (Arousal / Stres fizjologiczny)**: Silne ujemne wartości. Według modelu Hellera wskazuje to na potężne pobudzenie prawej kory ciemieniowej – pacjent jest przebodźcowany i działa w trybie ciągłej czujności wizualnej. Co fascynujące, największym "stresem" poznawczym jest dla niego bierne oglądanie trudnych treści, natomiast największą ulgę i spadek napięcia przynosi mu mechaniczne przewijanie głupot.

**O2/O1 (Strategia Przetwarzania Wzrokowego):**: Mózg domyślnie chłonie kolory i ruch, ignorując tekst. W fazie Smart Scroll jest jedyny moment w całym badaniu, w którym lewa kora wzrokowa (odpowiedzialna za analityczne czytanie) zostaje wreszcie zmuszona do ciężkiej pracy, aby nadążyć za przesuwanym tekstem lub trudnymi schematami.



## Scroll

In [ ]:
PROJECT_ROOT = Path.cwd().parent 

USER_MAP_PATH = PROJECT_ROOT / "EEG" / "EEG" / "user_map.json"
INTERACTIONS_PATH = PROJECT_ROOT / "EEG" / "research_events_rows.csv"
USERS_PATH = PROJECT_ROOT / "EEG" / "Users_rows.csv"

FILE_NAME = 'Case_07_ExG.csv'

In [ ]:
scroll_data = extract_scroll_events(FILE_NAME, INTERACTIONS_PATH, USERS_PATH, USER_MAP_PATH)

if scroll_data is not None and not scroll_data.empty:
    new_annotations = mne.Annotations(
        onset=scroll_data['onset'].values,
        duration=scroll_data['duration'].values,
        description=scroll_data['description'].values
    )

    raw_clean.set_annotations(new_annotations)
    
    print("SUCCESS: Annotations applied to raw_clean.")
    print(f"Time range of scrolls: {scroll_data['onset'].min():.2f}s to {scroll_data['onset'].max():.2f}s")
else:
    print("WARNING: No scroll data found. Plots will be empty.")

In [ ]:
compare_stages(
    raw_clean, 
    band='Beta', 
    fmin=13, 
    fmax=30, 
    channel='C3',  
    duration=DURATION
)

In [ ]:
# REWARD/MOTIVATION SYSTEM (Frontal Alpha)
# Channel: F3 (Left Frontal)
# Band: Alpha (8-12 Hz)
# Interpretation: THE LOWER THE GRAPH, THE GREATER THE “REWARD/ENGAGEMENT.”

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='F3',   # Key channel for positive/aspirational emotions
    duration=DURATION
)

In [ ]:
# ISUAL RELAXATION (Alpha)
# Channel: O1 (Back of the head)
# Band: Alpha (8-12 Hz)
# Interpretation: HIGH LINE = RELAXATION / ZOMBIE MODE. LOW = WORK.

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='O1', 
    duration=DURATION
)

In [ ]:
# INTELLECTUAL EFFORT (Frontal Theta) ---
# Channel: Fz (Center of forehead)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = STRONG THINKING/MEMORIZATION.

target_channel = 'Fz'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# COGNITIVE CONTROL & INHIBITION (Right Frontal Theta) ---
# Channel: F4 (Right forehead / Right Frontal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = SUSTAINED FOCUS & SUPPRESSING DISTRACTIONS (Mental effort to stay on task or regulate emotional impulses).

target_channel = 'F4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VERBAL & ANALYTICAL MEMORY (Left Parietal Theta) ---
# Channel: P3 (Left back of head / Left Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = PROCESSING TEXT/LANGUAGE (Reading captions, understanding speech, verbal working memory).

target_channel = 'P3'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VISUOSPATIAL PROCESSING & ATTENTION (Right Parietal Theta) ---
# Channel: P4 (Right back of head / Right Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = ENCODING VISUAL/SPATIAL INFO (Processing layout, movement, spatial memory, or complex imagery).

target_channel = 'P4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

## Podsumowanie
Najważniejsze odkrycie na tych wykresach wcale nie leży w samych falach mózgowych, ale w pionowych, przerywanych liniach, które oznaczają fizyczny ruch palca (Scroll Gesture).

**Brainrot Scroll:** Patrząc na jakikolwiek wykres z tej fazy (np. Beta C3), widzimy gęsty las czerwonych linii. Pacjent przewija ekran średnio co 5-10 sekund.

**Smart Scroll:** Są tylko dwa gesty scrolla w przeciągu pierwszych 100 sekund badania.

**Kora Ruchowa (Beta C3):**
Brainrot: Sygnał Beta (odpowiadający m.in. za planowanie i wykonanie ruchu) jest tu szarpany i poszarpany przez cały czas trwania okna, idealnie synchronizując się z ciągłym przewijaniem.

Smart: Na początku widać ogromny wystrzał Bety (do 20 µV), kiedy pacjent natrafia na film i wykonuje pierwszy scroll. Potem sygnał drastycznie opada i przez resztę czasu jest bardzo płaski i stabilny (trzyma się w wąskim paśmie wokół 4 µV). Kora ruchowa lewej półkuli została "wyłączona".

**Kora Czołowa i Ciemieniowa (Theta F4, P3)**:
Podczas ok. 57 sekundy: Na prawej korze czołowej (F4) pojawia się pik.
Dokładnie w tej samej sekundzie lewa kora ciemieniowa (P3 - Analiza Werbalna/Czytanie) wystrzeliwuje w górę.

W okolicach 57. sekundy filmu Smart, pacjent musiał przeczytać ze zrozumieniem wyjątkowo trudny komunikat, wykres lub tabelę. Nie było tam ruchu palcem (brak niebieskiej linii). Jego prawa kora czołowa (F4) zablokowała chęć ucieczki/przewinięcia dalej, zmuszając lewą półkulę (P3) do głębokiego przetworzenia tekstu.

**Kora Wzrokowa (Alpha O1):** Tryb Zombie vs Focus

* Brainrot: Widać gigantyczne wahania i wystrzały Alfy sięgające. Mózg jest zalewany obrazami, ale kora wzrokowa robi sobie ułamki sekund "drzemki", nie analizując ich głęboko.

* Smart: Piki Alfy wciąż występują (bo pacjent to wciąż pacjent z dominującą prawą półkulą), ale są rzadsze, a sygnał częściej spada do bardzo głębokich, twardych wartości.To momenty, gdy pacjent aktywnie skanuje litery na ekranie.

### Krótkie podsumowanie
Użytkownik nie potrafi "przewijać i uczyć się" jednocześnie.

Treści Brainrot pozwalają mu na zachowanie fizjologicznej rutyny – nerwowego stymulowania kciuka (ciągłe zrzuty Bety w C3), co nie obciąża jego pamięci werbalnej (P3 oscyluje chaotycznie bez wyraźnego celu). Gdy jednak trafia na treści Smart, jego organizm wchodzi w tryb skupienia. Zatrzymuje ruch (brak scrolli), kora czołowa z ogromnym wysiłkiem powstrzymuje impulsy (piki Theta na F4), a układ werbalny podejmuje ciężką walkę z tekstem (skoki na P3).